# Các Phương pháp Tối ưu hóa

Cho đến nay, bạn vẫn luôn sử dụng thuật toán Gradient Descent (Hạ độ dốc) để cập nhật các tham số và giảm thiểu hàm chi phí. Trong notebook này, bạn sẽ học cách sử dụng một số phương pháp tối ưu hóa nâng cao hơn, giúp đẩy nhanh quá trình học và có thể đạt được giá trị cuối cùng tốt hơn cho hàm chi phí. Việc sở hữu một thuật toán tối ưu hóa hiệu quả có thể tạo ra sự khác biệt lớn: thay vì phải chờ đợi hàng ngày trời, bạn chỉ mất vài giờ để có được kết quả tốt.

Sau khi hoàn thành notebook này, bạn sẽ có thể:

*   Áp dụng các phương pháp tối ưu hóa như (Stochastic) Gradient Descent, Momentum, RMSProp và Adam.
*   Sử dụng các minibatch ngẫu nhiên để đẩy nhanh tốc độ hội tụ và cải thiện quá trình tối ưu hóa.

Gradient descent hoạt động theo cơ chế "đi xuống dốc" trên hàm chi phí $J$. Hãy hình dung quá trình này giống như sau:
<img src="images/cost.jpg" style="width:650px;height:300px;">
<caption><center> <u> <b>Hình 1</b> </u>: <b>Giảm thiểu hàm chi phí cũng giống như việc tìm điểm thấp nhất trên một địa hình đồi núi</b><br> Tại mỗi bước huấn luyện, bạn cập nhật các tham số theo một hướng nhất định để cố gắng đạt đến điểm thấp nhất có thể. </center></caption>

**Ký hiệu**: Như thường lệ, $\frac{\partial J}{\partial a } = $ `da` đối với bất kỳ biến `a` nào.

Chúng ta cùng bắt đầu nhé!

## Lưu ý quan trọng về việc nộp bài cho hệ thống chấm điểm tự động (AutoGrader)

Trước khi nộp bài tập cho hệ thống chấm điểm tự động, vui lòng đảm bảo bạn không thực hiện các hành động sau:

1.  Thêm bất kỳ câu lệnh `print` _bổ sung_ nào vào bài tập.
2.  Thêm bất kỳ ô mã (code cell) _bổ sung_ nào vào bài tập.
3.  Thay đổi bất kỳ tham số nào của hàm.
4.  Sử dụng các biến toàn cục (global variables) bên trong các bài tập được chấm điểm. Trừ khi có hướng dẫn cụ thể, vui lòng tránh sử dụng biến toàn cục và thay vào đó hãy sử dụng biến cục bộ (local variables).
5.  Thay đổi mã nguồn của bài tập ở những phần không yêu cầu, chẳng hạn như tạo thêm các biến _bổ sung_. Nếu thực hiện bất kỳ thao tác nào dưới đây, bạn sẽ gặp lỗi kiểu như `Grader not found` (hoặc một lỗi bất thường tương tự) khi nộp bài tập. Hãy kiểm tra các vấn đề này trước khi tìm kiếm sự trợ giúp hoặc sửa lỗi trong bài tập của mình. Nếu gặp tình trạng này và không nhớ rõ những thay đổi đã thực hiện, bạn có thể lấy lại bản sao mới của bài tập bằng cách làm theo các [hướng dẫn](https://www.coursera.org/learn/deep-neural-network/supplement/QWEnZ/h-ow-to-refresh-your-workspace) này.

## Table of Contents
- [1- Packages](#1)
- [2 - Gradient Descent](#2)
    - [Exercise 1 - update_parameters_with_gd](#ex-1)
- [3 - Mini-Batch Gradient Descent](#3)
    - [Exercise 2 - random_mini_batches](#ex-2)
- [4 - Momentum](#4)
    - [Exercise 3 - initialize_velocity](#ex-3)
    - [Exercise 4 - update_parameters_with_momentum](#ex-4)
- [5 - Adam](#5)
    - [Exercise 5 - initialize_adam](#ex-5)
    - [Exercise 6 - update_parameters_with_adam](#ex-6)
- [6 - Model with different Optimization algorithms](#6)
    - [6.1 - Mini-Batch Gradient Descent](#6-1)
    - [6.2 - Mini-Batch Gradient Descent with Momentum](#6-2)
    - [6.3 - Mini-Batch with Adam](#6-3)
    - [6.4 - Summary](#6-4)
- [7 - Learning Rate Decay and Scheduling](#7)
    - [7.1 - Decay on every iteration](#7-1)
        - [Exercise 7 - update_lr](#ex-7)
    - [7.2 - Fixed Interval Scheduling](#7-2)
        - [Exercise 8 - schedule_lr_decay](#ex-8)
    - [7.3 - Using Learning Rate Decay for each Optimization Method](#7-3)
        - [7.3.1 - Gradient Descent with Learning Rate Decay](#7-3-1)
        - [7.3.2 - Gradient Descent with Momentum and Learning Rate Decay](#7-3-2)
        - [7.3.3 - Adam with Learning Rate Decay](#7-3-3)
    - [7.4 - Achieving similar performance with different methods](#7-4)

<a name='1'></a>
## 1- Packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import math
import sklearn
import sklearn.datasets

from opt_utils_v1a import load_params_and_grads, initialize_parameters, forward_propagation, backward_propagation
from opt_utils_v1a import compute_cost, predict, predict_dec, plot_decision_boundary, load_dataset
from copy import deepcopy
from testCases import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2